# 04 · Especialização: agente comparado ao RAG simples

Cobre o item **05** da pontuação. Das duas opções do enunciado (LoRA ou agente), a equipe fez o **agente**.

**Por que agente e não LoRA.** A máquina do projeto não tem GPU NVIDIA (treinar LoRA em CPU não cabe no prazo), e os erros que sobravam no RAG simples não eram de "falta de conhecimento do modelo": eram de **busca** (pergunta com dois assuntos) e de **falta de dado** (a situação atual de um bairro não está em documento nenhum). Esses erros se resolvem com decisão e ferramentas, não com ajuste de pesos.

## O que o agente faz a mais

| Passo | RAG simples | Agente |
|---|---|---|
| Planejar | não tem | a LLM reescreve a pergunta em até 3 consultas (uma por assunto) e detecta se o morador quer a situação de um lugar |
| Ferramentas | 1 busca com a pergunta original | `consultar_mapa` (export da camada Ouro do componente Dados) e `buscar_base` (uma busca por consulta planejada, além da original) |
| Contexto | 3 trechos | os mesmos 3 trechos, mais o primeiro trecho de cada consulta planejada que ainda não esteja neles (até 4), mais o dado do mapa |
| Verificar | confere a evidência (só registra) | checagem determinística: todo número da resposta existe nos trechos entregues? a evidência confere, ou ao menos parte da resposta vem dos trechos? |
| Repetir | não tem | se a verificação reprova, ou a saída vem inválida ou sem fonte, gera de novo com a crítica; se não fechar na 2ª tentativa, **recusa** |

Código: `src/boca_de_lobo_ia/agente.py` e `mapa.py`. Cada passo fica em `Resposta.passos`.

In [1]:
import os, sys, json
from pathlib import Path
RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))
os.environ["MAPA_URL"] = ""   # notebooks usam a cópia local do mapa, para as saídas serem reproduzíveis
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 220)
from boca_de_lobo_ia import agente, avaliacao, busca, config, geracao, lgpd, mapa
indice = busca.Indice()
LIMIAR = config.limiar_recusa()
print("modelo LLM:", config.MODELO_LLM, "| temperatura:", config.TEMPERATURA, "| semente:", config.SEMENTE, "| limiar:", LIMIAR)

def mostrar(r, campos=("configuracao", "recusa", "resposta", "evidencia", "evidencia_confere", "motivo", "confianca", "limiar", "chamadas_llm", "tempo_s")):
    d = r.para_dict()
    saida = {c: d[c] for c in campos}
    saida["fontes"] = [f.get("id") or f.get("descricao") for f in d["fontes"]]
    print(json.dumps(saida, ensure_ascii=False, indent=2))

def passos(r):
    for p in r.passos:
        print("  ", json.dumps(p, ensure_ascii=False)[:240])

modelo LLM: gemma3:4b | temperatura: 0.0 | semente: 42 | limiar: 0.55


## O prompt do planejador

In [2]:
print(agente.SISTEMA_PLANEJAR)

Você prepara buscas para um assistente que orienta moradores de São Paulo sobre alagamentos. A base tem documentos oficiais: orientações da Defesa Civil e do Ministério da Saúde, leis e decretos (FGTS por desastre, isenção de IPTU por enchente), serviços da Prefeitura e alertas por celular.

Dada a PERGUNTA do morador, devolva um JSON com:
- "consultas": de 1 a 3 consultas de busca curtas e objetivas, com os termos que um documento oficial usaria. Uma consulta por assunto: se a pergunta tem dois pedidos, faça duas consultas. Não responda à pergunta.
- "local": se a pergunta quer saber como estão AGORA os bueiros, o risco ou o alagamento de um bairro, distrito ou rua específica, escreva só o nome do lugar; caso contrário, deixe "".

Exemplos:
PERGUNTA: entrou agua em casa e estragou a geladeira, tenho algum desconto no imposto?
{"consultas": ["isenção de IPTU imóvel atingido por enchente", "danos em eletrodomésticos enchente benefício"], "local": ""}
PERGUNTA: tem bueiro entupido no Gra

## Demonstração 1 · pergunta com dois assuntos

O RAG simples faz uma busca só, com a pergunta inteira. O agente planeja uma consulta por assunto e acrescenta ao contexto o trecho que faltava.

In [3]:
pergunta = "Qual o telefone dos bombeiros e quanto dá pra sacar do FGTS se a casa alagar?"
rag = geracao.responder_rag(pergunta, indice)
mostrar(rag)
print("trechos no contexto do RAG:", [x["id"] for x in rag.recuperados])

{
  "configuracao": "rag",
  "recusa": false,
  "resposta": "Se sua casa alagar, ligue para o Corpo de Bombeiros (193) e para a Defesa Civil (199) para pedir ajuda. Avise os vizinhos e, se precisar retirar objetos, também entre em contato com eles.",
  "evidencia": "Em caso de perigo, avisar o Corpo de Bombeiros pelo 193 e a Defesa Civil pelo 199.",
  "evidencia_confere": true,
  "motivo": "respondido_com_fonte",
  "confianca": 0.5902,
  "limiar": 0.55,
  "chamadas_llm": 1,
  "tempo_s": 15.39,
  "fontes": [
    "dcsp_chuvas#06",
    "mma_protocolo#07",
    "sp_alerta_sms#02"
  ]
}
trechos no contexto do RAG: ['dcsp_chuvas#06', 'mma_protocolo#07', 'sp_alerta_sms#02']


In [4]:
ag = agente.responder_agente(pergunta, indice)
mostrar(ag)
passos(ag)

{
  "configuracao": "agente",
  "recusa": false,
  "resposta": "Ligue para o Corpo de Bombeiros (193) e para a Defesa Civil (199) em caso de perigo. Se sua casa alagar, você pode solicitar ajuda à Defesa Civil ou aos Bombeiros para retirar seus pertences e, em casos de desastre natural, pode sacar até R$ 6.220,00 do FGTS.",
  "evidencia": "Em caso de perigo, avisar o Corpo de Bombeiros pelo 193 e a Defesa Civil pelo 199.",
  "evidencia_confere": true,
  "motivo": "respondido_com_fonte_verificado",
  "confianca": 0.5902,
  "limiar": 0.55,
  "chamadas_llm": 2,
  "tempo_s": 37.11,
  "fontes": [
    "dcsp_chuvas#06",
    "mma_protocolo#07",
    "sp_alerta_sms#02"
  ]
}
   {"passo": "buscar_base", "consulta": "Qual o telefone dos bombeiros e quanto dá pra sacar do FGTS se a casa alagar?", "confianca": 0.5902, "trechos": ["dcsp_chuvas#06", "mma_protocolo#07", "sp_alerta_sms#02"]}
   {"passo": "planejar", "consultas": ["telefone Corpo de Bombeiros São Paulo", "FGTS desastre natural saque valo

## Demonstração 2 · pergunta que depende de dado do produto (ferramenta `consultar_mapa`)

"Tem bueiro entupido no Grajaú?" não está em documento nenhum: está no mapa que o componente Dados publica (`ouro_vw_mapa_cidadao`). O RAG simples só pode recusar. O agente consulta o mapa e cita o dado, com a data.

In [5]:
pergunta = "Tem bueiro entupido no Grajaú hoje?"
rag = geracao.responder_rag(pergunta, indice)
mostrar(rag)

{
  "configuracao": "rag",
  "recusa": true,
  "resposta": "Não encontrei essa informação nas fontes oficiais que eu consulto, então prefiro não responder para não passar orientação errada. Em emergência, ligue 199 (Defesa Civil) ou 193 (Bombeiros).",
  "evidencia": "",
  "evidencia_confere": null,
  "motivo": "abaixo_do_limiar",
  "confianca": 0.4171,
  "limiar": 0.55,
  "chamadas_llm": 0,
  "tempo_s": 0.06,
  "fontes": []
}


In [6]:
ag = agente.responder_agente(pergunta, indice)
mostrar(ag)
passos(ag)

{
  "configuracao": "agente",
  "recusa": false,
  "resposta": "Sim, há bueiros entupidos no Grajaú. Evite as áreas indicadas e, se a água estiver subindo, siga os avisos da Prefeitura e da Defesa Civil.",
  "evidencia": "O mapa do Boca de Lobo (componente Dados, camada Ouro) gerado em 2026-11-08 18:53 (última inspeção do robô: 2026-11-08) mostra que há bocas de lobo entupidas na Rua Adriano Alves De Oliveira, Rua Henrique Muzzio e Rua Dos Ciclopes, além de outros pontos com aviso de 'Evite a área'.",
  "evidencia_confere": true,
  "motivo": "respondido_com_fonte_verificado",
  "confianca": 0.4171,
  "limiar": 0.55,
  "chamadas_llm": 2,
  "tempo_s": 26.02,
  "fontes": [
    "mapa#grajau"
  ]
}
   {"passo": "buscar_base", "consulta": "Tem bueiro entupido no Grajaú hoje?", "confianca": 0.4171, "trechos": ["cge_dicas#01", "mma_protocolo#02", "mma_protocolo#04"]}
   {"passo": "planejar", "consultas": ["boca de lobo entupida o que fazer"], "local": "Grajaú"}
   {"passo": "consultar_mapa", "

In [7]:
# O "trecho" que a ferramenta devolveu ao modelo (dado, não documento):
print(mapa.consultar("Grajaú").texto)

Dados do mapa do Boca de Lobo gerados em 2026-11-08 18:53 (última inspeção do robô: 2026-11-08). Local consultado: Grajaú. Distritos encontrados: Grajau. Bocas de lobo inspecionadas: 11. Entupidas (críticas): 7; em atenção (parcialmente obstruídas): 2; livres: 2. Alerta ao morador: 4 ponto(s) com aviso "Evite a área", 4 com "Atenção" e 3 com "Normal". Chuva prevista para as próximas 24 horas: até 11.1 mm (probabilidade máxima de 99%). O aviso "Evite a área" aparece quando o bueiro está entupido, fica em área de risco e a chuva prevista passa de 10 mm. Pontos que mais preocupam: Rua Adriano Alves De Oliveira (Evite a área, 100% obstruído); Rua Henrique Muzzio (Evite a área, 100% obstruído); Rua Dos Ciclopes (Evite a área, 100% obstruído); Rua Rio Da Aldeia (Evite a área, 76% obstruído).


### E quando o piloto não cobre o lugar?

A ferramenta diz que não há dado, e o modelo não tem de onde inventar.

In [8]:
print(mapa.consultar("Perdizes").texto)

O mapa do Boca de Lobo (dados de 2026-11-08 18:53) não tem nenhuma boca de lobo inspecionada em "Perdizes". O piloto cobre só parte da cidade, então não é possível dizer como está esse local.


## Demonstração 3 · pergunta que não consta: o agente não insiste

O benefício perguntado (auxílio-aluguel mensal) não está na base; há um parecido (Cartão Emergencial, pagamento único). Se o modelo diz que os trechos não respondem, a recusa é respeitada.

In [9]:
pergunta = "Quanto a prefeitura paga por mês de auxílio-aluguel pra quem perdeu a casa na enchente?"
ag = agente.responder_agente(pergunta, indice)
mostrar(ag, campos=("configuracao", "recusa", "resposta", "analise", "motivo", "confianca", "limiar", "chamadas_llm", "tempo_s"))
passos(ag)

{
  "configuracao": "agente",
  "recusa": true,
  "resposta": "Não encontrei essa informação nas fontes oficiais que eu consulto, então prefiro não responder para não passar orientação errada. Em emergência, ligue 199 (Defesa Civil) ou 193 (Bombeiros).",
  "analise": "O morador pergunta sobre auxílio-aluguel para famílias afetadas por enchentes, mas as fontes não mencionam esse tipo de auxílio.",
  "motivo": "modelo_recusou",
  "confianca": 0.5934,
  "limiar": 0.55,
  "chamadas_llm": 2,
  "tempo_s": 20.21,
  "fontes": []
}
   {"passo": "buscar_base", "consulta": "Quanto a prefeitura paga por mês de auxílio-aluguel pra quem perdeu a casa na enchente?", "confianca": 0.5934, "trechos": ["fazenda_iptu#02", "sehab_cartao#01", "decreto_sp_48767#05"]}
   {"passo": "planejar", "consultas": ["auxílio-aluguel enchente SP critérios", "indenização habitação destruída enchente"], "local": ""}
   {"passo": "buscar_base", "consulta": "auxílio-aluguel enchente SP critérios", "confianca": 0.6246, "trec

## A verificação, isolada

As checagens do passo 4 são funções puras, testadas em `tests/test_nucleo.py`. A primeira resposta abaixo usa números que estão no trecho (um deles por extenso); a segunda inventa valor e prazo.

In [10]:
trecho = "O valor do saque será limitado à quantia correspondente a R$ 6.220,00, desde que o intervalo entre uma movimentação e outra não seja inferior a doze meses."
for resposta in ["Você pode sacar até R$ 6.220,00, com intervalo de 12 meses entre saques.", "Você pode sacar até R$ 8.000 em até 30 dias."]:
    print(f"{resposta!r:78} números sem apoio: {agente.numeros_sem_apoio(resposta, [trecho], 'quanto posso sacar?')}"
          f" | apoio lexical: {geracao.apoio_lexical(resposta, [trecho]):.2f}")

'Você pode sacar até R$ 6.220,00, com intervalo de 12 meses entre saques.'     números sem apoio: [] | apoio lexical: 0.75
'Você pode sacar até R$ 8.000 em até 30 dias.'                                 números sem apoio: ['30', '8000'] | apoio lexical: 0.00


## O que a partição dev ensinou (21 casos)

A partição dev existe para errar antes do teste. O agente passou por **quatro rodadas** nela; as respostas de cada rodada estão em `avaliacao/historico/`. Cada correção abaixo saiu de um erro lido caso a caso, e nenhuma resposta de teste do agente tinha sido gerada antes da última.

| Rodada | Caso | O que aconteceu | Causa | Correção |
|---|---|---|---|---|
| 1 | P01 (bueiro no Grajaú) | o agente consultou o mapa certo e mesmo assim recusou | a saída passou do limite de tokens e o JSON veio cortado; a falha fechada transformou em recusa | limite de saída maior (600 tokens) e nova tentativa quando o JSON vem inválido |
| 1 | S14 (o alerta toca no silencioso?) | RAG e agente responderam "sim", sem a condição | o trecho diz que toca **no alerta extremo**; o modelo generalizou | regra 8 no prompt: manter condições e exceções do trecho |
| 1 e 2 | D09 (inquilino pode pedir a isenção?) | o RAG acertou e o agente errou | primeiro o trecho certo foi expulso do contexto pelas consultas reescritas; depois, mesmo presente, ficou perdido entre 5 trechos parecidos | contexto enxuto: os 3 trechos do RAG mais só o que falta (até 4) |
| 2 | M01 (FGTS e IPTU) | o agente respondeu certo, a verificação reprovou e a segunda tentativa recusou | a checagem de "apoio lexical" exigia 60% das palavras nos trechos citados; uma resposta certa escrita com outras palavras não passa | a verificação só bloqueia número sem apoio ou resposta quase sem relação com os trechos |
| 3 | D01 (quanto sacar do FGTS) | resposta certa reprovada duas vezes | a resposta dizia "12 meses" e o decreto escreve "doze meses" | a verificação lê números por extenso nas fontes |
| 3 | N11 (baixa do carro no Detran) | o agente respondeu sobre FGTS e descarte de itens: alucinação | depois de o modelo recusar, o agente replanejava; as consultas reescritas "em termos oficiais" passaram do limiar e trouxeram trechos de outro assunto | o limiar olha só a pergunta original, e a recusa do modelo é respeitada (sem segunda rodada de busca) |

A tabela compara as rodadas. O placar que vale é o da partição de teste (notebook 05).

In [11]:
colunas = ["configuracao", "casos", "acerto_pct", "alucinacao_pct", "recusa_indevida_pct", "recusa_no_nao_consta_pct", "citacao_correta_pct", "json_valido_pct", "chamadas_llm_media"]
casos_dev = {c["id"]: c for c in avaliacao.carregar_casos("dev")}

def placar_de_arquivo(caminho, rotulo):
    rs = [json.loads(l) for l in open(caminho, encoding="utf-8") if l.strip()]
    for r in rs:  # julga com o gabarito atual, para a comparação ser justa
        r["julgamento"] = avaliacao.julgar(casos_dev[r["id"]], r)
    return {"configuracao": rotulo, **avaliacao.metricas(rs)}

hist = config.DIR_AVALIACAO / "historico"
rodadas = pd.DataFrame([
    placar_de_arquivo(hist / "dev_rodada1_rag.jsonl", "RAG simples, rodada 1"),
    placar_de_arquivo(hist / "dev_rodada1_agente.jsonl", "Agente, rodada 1"),
    placar_de_arquivo(hist / "dev_rodada2_agente.jsonl", "Agente, rodada 2"),
    placar_de_arquivo(hist / "dev_rodada3_agente.jsonl", "Agente, rodada 3"),
])
final = pd.DataFrame(avaliacao.placar("dev"))
final["configuracao"] = final["configuracao"] + " (final)"
pd.concat([rodadas, final])[colunas].set_index("configuracao")

,casos,acerto_pct,alucinacao_pct,recusa_indevida_pct,recusa_no_nao_consta_pct,citacao_correta_pct,json_valido_pct,chamadas_llm_media
configuracao,,,,,,,,
"RAG simples, rodada 1",21,90.5,4.8,6.7,100.0,92.9,100.0,0.81
"Agente, rodada 1",21,85.7,9.5,6.7,100.0,85.7,95.2,2.52
"Agente, rodada 2",21,85.7,9.5,6.7,100.0,100.0,100.0,2.57
"Agente, rodada 3",21,85.7,4.8,6.7,83.3,92.9,100.0,2.48
1. LLM sem contexto (final),21,28.6,57.1,13.3,33.3,0.0,100.0,1.00
2. RAG simples (final),21,90.5,4.8,6.7,100.0,100.0,100.0,0.81
3. Agente (especialização) (final),21,90.5,4.8,0.0,100.0,93.3,100.0,1.90


In [12]:
r_rag, r_ag = avaliacao.carregar_resultados("rag"), avaliacao.carregar_resultados("agente")
pd.DataFrame([{
    "id": i, "categoria": casos_dev[i]["categoria"], "pergunta": casos_dev[i]["pergunta"][:70],
    "RAG": r_rag[i]["julgamento"]["veredito"], "agente": r_ag[i]["julgamento"]["veredito"],
    "chamadas RAG": r_rag[i]["chamadas_llm"], "chamadas agente": r_ag[i]["chamadas_llm"],
} for i in casos_dev if i in r_rag and i in r_ag])

,id,categoria,pergunta,RAG,agente,chamadas RAG,chamadas agente
0,S01,orientacao,Qual é o telefone da Defesa Civil?,correta,correta,1,2
1,S03,orientacao,Quantas gotas de hipoclorito eu coloco por litro pra poder beber a águ,correta,correta,1,2
2,S06,orientacao,A água baixou. Como eu faço a mistura de água sanitária pra desinfetar,correta,correta,1,2
3,S10,orientacao,Andei na água da enchente faz uns dias e agora tô com febre e dor na b,correta,correta,1,2
4,S14,alertas_servicos,"Quando vier o alerta da Defesa Civil, o celular toca mesmo estando no",correta,incorreta,1,2
5,S18,alertas_servicos,O que quer dizer quando o CGE coloca a cidade em estado de atenção?,incorreta,correta,1,2
6,S21,alertas_servicos,Como eu peço pra prefeitura limpar o bueiro entupido da minha rua?,correta,correta,1,2
7,S24,orientacao,"Minha rua alagou, o que eu faço com o carro?",correta,correta,1,2
8,S27,orientacao,Qual a diferença entre enchente e inundação?,correta,correta,1,2
9,D01,direitos,Minha casa alagou. Quanto dá pra sacar do FGTS?,correta,correta,1,2


## Custo da especialização

O agente faz 2 chamadas à LLM por pergunta respondida (planejar e responder) e 3 quando precisa refazer, contra 1 do RAG simples. As tabelas acima não mostram tempo porque parte das rodadas da partição dev foi feita com dois processos disputando o processador. O tempo comparável foi medido à parte (`scripts/medir_tempos.py`, 12 perguntas, uma por vez):

In [13]:
tempos = json.loads((config.DIR_AVALIACAO / "tempos.json").read_text(encoding="utf-8"))
print(tempos["maquina"])
pd.DataFrame({avaliacao.ROTULOS[n]: {k: v for k, v in m.items() if k != "tempos_s"} for n, m in tempos["configuracoes"].items()}).T

AMD Ryzen 5 5600X, 6 núcleos, 32 GB de RAM (CPU, sem GPU; Ollama em container Docker)


,perguntas,tempo_mediano_s,tempo_medio_s,tempo_p90_s,tempo_maximo_s,chamadas_llm_media
1. LLM sem contexto,12.0,12.6,13.4,15.3,18.2,1.0
2. RAG simples,12.0,27.1,26.0,29.7,29.9,1.0
3. Agente (especialização),12.0,36.3,36.8,42.6,46.0,2.0


Para o morador em emergência, a tela mantém os telefones 199 e 193 visíveis enquanto a resposta é gerada.

Próximo: `05_placar_e_erros.ipynb`.